In [0]:
df_silver_order=spark.read.format('delta').table(
    'freshcart_team1.silver_data_orders'
)

df_silver_order.show(5)

In [0]:
#  create gold tables 

spark.sql("""
CREATE OR REPLACE TABLE freshcart_team1.gold_city_revenue
USING DELTA
AS

WITH city_metrics AS (
    SELECT
        city,
        COUNT(order_id) as total_orders,
        SUM(total_amount)     as total_revenue,
        AVG(total_amount)  as avg_order_value
    FROM freshcart_team1.silver_orders
    GROUP BY city
)

SELECT * FROM city_metrics
""")



spark.sql("""
OPTIMIZE freshcart_team1.gold_city_revenue
ZORDER BY (city)
""")

In [0]:
from delta.tables import DeltaTable

df_gold_city_rev=spark.read.format('delta').table(
    'freshcart_team1.gold_city_revenue'
)

dt=DeltaTable.forName(spark,"freshcart_team1.gold_city_revenue")

dt.alias('target').merge(
    df_gold_city_rev.alias('source'),
    'target.city=source.city'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute(
)

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE de_workspace26.freshcart_team1.gold_product_performance
USING DELTA
AS

WITH joined_data AS (
    SELECT
        oi.product_id,
        p.category,
        oi.quantity,
        oi.line_total
    FROM freshcart_team1.silver_order_items oi
    INNER JOIN freshcart_team1.silver_products p
        ON oi.product_id = p.product_id
),

category_metrics AS (
    SELECT
        category,
        SUM(quantity) AS total_units_sold,
        SUM(line_total)   AS total_revenue
    FROM joined_data
    GROUP BY category
)

SELECT * FROM category_metrics
""")

In [0]:
from delta.tables import DeltaTable

df_gold_prod_perf=spark.read.format('delta').table(
    'freshcart_team1.gold_product_performance'
)

dt=DeltaTable.forName(spark,"freshcart_team1.gold_product_performance")

dt.alias('target').merge(
    df_gold_prod_perf.alias('source'),
    'target.category=source.category'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE freshcart_team1.gold_delivery_sla
USING DELTA
AS

WITH delivery_metrics AS (
    SELECT
        city,
        COUNT(*) AS total_deliveries,
        SUM(CASE 
                WHEN delivery_minutes <= 15 THEN 1 
                ELSE 0 
            END) AS within_sla
    FROM freshcart_team1.silver_deliveries
    GROUP BY city
),

sla_calc AS (
    SELECT
        city,
        total_deliveries,
        within_sla,
        ROUND((within_sla * 100.0 / total_deliveries), 2) AS sla_percentage
    FROM delivery_metrics
)

SELECT * FROM sla_calc
""")

In [0]:
from delta.tables import DeltaTable

df_gold_delivery_sla=spark.read.format('delta').table(
    'freshcart_team1.gold_delivery_sla'
)

dt= DeltaTable.forName(spark,"freshcart_team1.gold_delivery_sla")

dt.alias('target').merge(
    df_gold_delivery_sla.alias('source'),
    'target.city=source.city'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute(
)

In [0]:
spark.sql("""
OPTIMIZE freshcart_team1.gold_product_performance
ZORDER BY (category)
""")


spark.sql("""
OPTIMIZE freshcart_team1.gold_delivery_sla
ZORDER BY (city)
""")